# 05 · Scaling up

**pragmatiq series, notebook 5.** The series ran a small sample (4,000 users
by default) against a checkpoint trained elsewhere: final-sweep candidate D,
3,000 pretraining steps fine-tuned at LoRA rank 16. This notebook shows the
configs behind that checkpoint and what the recorded runs say about more
pretraining, a larger LoRA rank, and a longer context. It then gives the
commands to reproduce the checkpoint, to package your own for these
notebooks, and to swap in another dataset.

It reads only committed files, needs no GPU, and runs in seconds. Every
number comes from two records made on one GB10: the
[final sweep](../../benchmarks/results/blueprint_v2_final_sweep.md) and the
earlier [tuning search](../../benchmarks/results/blueprint_v2_hybrid_gb10_tuning.md).

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tfm.config import load_recipe_config
from tfm.runtime.common import recipe_dir, repo_root
from tfm.training.reproducibility import seed_everything

GREEN, DARK, GREY = "#76B900", "#1A1A1A", "#8C8C8C"
PALETTE = [GREEN, "#0074DF", "#F2A900", "#5D1682", "#E52020", DARK, GREY, "#00A3A3"]
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
})

RECIPE = recipe_dir("blueprint_v2")
config = load_recipe_config(RECIPE / "config.yaml")
seed_everything(config.seed)
tuning = json.loads((repo_root() / "benchmarks/results/blueprint_v2_hybrid_gb10_tuning.json").read_text())
final_sweep = json.loads((repo_root() / "benchmarks/results/blueprint_v2_final_sweep.json").read_text())

## The recipe's configs

One recipe, three sizes, and two ablations of it. Each is a committed YAML
file with an explicit seed, and every slot of the model is one key in it.
Another,
`benchmarks/ablations/blueprint_v2_lab_v1_flat.yaml`, runs v1's flat decoder on the lab rows for
comparison.

In [ ]:
rows = {}
LAB = repo_root() / "benchmarks" / "ablations" / "blueprint_v2_lab.yaml"
for name, path in {
    "config.smoke.yaml": RECIPE / "config.smoke.yaml",
    "ablation: lab (untuned)": LAB,
    "config.yaml": RECIPE / "config.yaml",
    "ablation: final-sweep D (the checkpoint)": repo_root() / "benchmarks" / "ablations" / "blueprint_v2_final_d.yaml",
    "config.scale.yaml": RECIPE / "config.scale.yaml",
}.items():
    c = load_recipe_config(path)
    g = c.model.composition.granularity
    rows[name] = {
        "users sampled": c.data.max_entities or c.data.n_entities,
        "events per row": c.data.sequence_length,
        "hidden": g.decoder.hidden_size,
        "depths (profile/event/decoder)": f"{g.profile_depth}/{g.event_depth}/{g.decoder.depth}",
        "hashed keys": ", ".join(c.features.hashed_keys) or "-",
        "token cache": "yes" if c.data.token_cache else "no",
        "optimizer": c.training.optimizer.name,
        "steps x batch": f"{c.training.max_steps} x {c.training.batch_size}",
        "LoRA rank": c.fraud_task.finetune.rank if c.fraud_task.finetune.enabled else "-",
        "precision": c.training.precision,
    }
pd.DataFrame(rows).T

- `config.smoke.yaml` is the CPU fixture that the recipe smoke test runs.
- `benchmarks/ablations/blueprint_v2_lab.yaml` is the untuned lab model: byte-level identifiers, one
  row per sampled user.
- `config.yaml` is the recipe's reference setting, from the tuning search:
  the same model and probes, with hashed identifiers, pretraining on every
  128-event window of all 100,000 users (read from a token cache), Muon,
  1,500 steps, and LoRA rank 8. `pragma_like` is matched to it.
- `benchmarks/ablations/blueprint_v2_final_d.yaml` made the series'
  checkpoint: `config.yaml` with 3,000 steps (warmup 150) and LoRA rank 16,
  the final sweep's winner.
- `config.scale.yaml` is the ~100M-parameter model. It carries the tuned
  settings over and trains with DDP on several GPUs. No run of it is recorded
  yet.

## What more pretraining and a larger rank bought

The final sweep trained `config.yaml`'s model for 1,500 steps (A) and 3,000
steps (B), and fine-tuned each checkpoint with LoRA at rank 8 and at rank 16
(C reuses A's checkpoints, D reuses B's), at three seeds on 10,000 probe
users. A rule written down before the first run chose D. On its own, more
pretraining (B) raised the median fraud lift by 12 points but lowered the
transaction-level one, and a larger rank (C) moved no median by more than 3
points. Together, D added 11 to 19 points over A on fraud,
transaction-level fraud, and churn. LTV barely moved. The ranges are wide: D's fraud lift is +89% at
one seed and +163% to +183% at the other two.

In [ ]:
arms = final_sweep["selection"]["arms"]
TASKS = {"fraud": "fraud, per user", "fraud_txn": "fraud, per txn", "churn_6m": "churn 6m",
         "ltv_positive": "LTV positive (ROC-AUC)"}
CANDIDATES = {"A": ("A: 1,500 steps, rank 8", GREY), "B": ("B: 3,000 steps, rank 8", "#0074DF"),
              "C": ("C: 1,500 steps, rank 16", "#F2A900"),
              "D": ("D: 3,000 steps, rank 16 (the checkpoint)", GREEN)}
fig, ax = plt.subplots(figsize=(11, 3.8))
x = np.arange(len(TASKS))
for i, (name, (label, colour)) in enumerate(CANDIDATES.items()):
    lifts = [100 * np.array(arms[name]["lora"]["lifts"][task]) for task in TASKS]
    median = np.array([np.median(seeds) for seeds in lifts])
    spread = np.array([median - [s.min() for s in lifts], [s.max() for s in lifts] - median])
    ax.bar(x + (i - 1.5) * 0.2, median, 0.2, yerr=spread, capsize=3, color=colour, label=label)
ax.set_xticks(x, list(TASKS.values()))
ax.set_ylabel("LoRA lift over raw-feature XGBoost (%)")
ax.set_title("Final sweep, recorded: median and range over three seeds")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(final_sweep["selection"]["winner"], "won:", final_sweep["selection"]["reason"])

## What a longer context bought

The model reads the last 128 events of each history. With 512, the
raw + embedding arm barely moved on any task (left). A logistic probe on the
embedding plus surprise moved more, in opposite directions (right):
`fraud_txn` rose and `fraud_user` fell. 128 was kept. (The tuning search's
proxy, on the 1,500-step model: one probe split, 8 XGBoost trials.)

In [ ]:
tasks = {"fraud_txn": "history_window", "churn_6m": "churn_6m", "fraud_user (search only)": "fraud"}
x = np.arange(len(tasks))
short = next(r for r in tuning["search"]["stage3"] if r["name"] == "causal")["pr_auc"]
long = next(r for r in tuning["search"]["stage4"] if r["name"] == "causal_512")["pr_auc"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for ax, (metric, title) in zip(axes, [("xgboost_raw_embedding", "raw + embedding (XGBoost)"),
                                       ("last_surprise", "embedding + surprise (logistic probe)")]):
    ax.bar(x - 0.18, [short[f"{t}/{metric}"] for t in tasks.values()], 0.36, color=GREEN, label="128 events")
    ax.bar(x + 0.18, [long[f"{t}/{metric}"] for t in tasks.values()], 0.36, color="#0074DF", label="512 events")
    ax.set_xticks(x, list(tasks), fontsize=8)
    ax.set_title(title)
axes[0].set_ylabel("PR-AUC (one split)")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Reproduce the lab checkpoint

On one GB10, building the token cache takes about 4 minutes and pretraining
about 42 (the recipe's 1,500 steps take about 24). The GB10's memory is unified, and its caching allocator fragments
on variable-length batches. `expandable_segments` holds the run near the
22 GB it allocates.

```bash
export TFM_PRAGMATIQ_ROOT=/path/to/synth_100k_gen2_m31_seed0
uv run tfm cache-tokens --recipe blueprint_v2 --workers 6
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
uv run tfm report --config benchmarks/ablations/blueprint_v2_final_d.yaml \
  --device cuda --run-dir outputs/ablations/blueprint_v2_final_d
```

`tfm report` trains, embeds, evaluates, and writes a model card. Its
evaluation scores the `last_token` readout, the surprise arm, and LoRA on
every task, with the notebooks' fixed XGBoost setting.
`benchmarks/blueprint_v2_final_sweep.sh` reruns all four sweep candidates at
three seeds. The recorded lift, with surprise
and three probe splits, comes from `benchmarks/final_eval.py`. With the
notebooks' fixed XGBoost setting and their four tasks (`history_window` is
`fraud_txn`), it takes about 3.5 minutes per split seed; add `--random-init`
for the untrained control. `--finetune recipes/blueprint_v2/config.yaml`
adds the LoRA arm on the same rows, at about 11 minutes per split seed:

```bash
uv run python benchmarks/final_eval.py --run-dir <run dir holding seed_config.yaml> \
  --out outputs/final_eval.json --split-seeds 0 1 2 --fixed-xgboost \
  --tasks history_window churn_6m ltv_positive default_12m
```

Without `--fixed-xgboost` it searches XGBoost per arm (`--xgboost-trials 30`)
and adds the hand-built reference arms, as the tuning record's headline
does; that takes about 6.5 minutes per split seed.

To build the book itself, see the
[recipe README](../../recipes/blueprint_v2/README.md).

## Use your own checkpoint in these notebooks

`scripts/package_lab_assets.py` bundles a run's `model.pt` with the
tokenizer it was trained with and the config (by default candidate D's). It writes a directory that
`TFM_V2_CHECKPOINT` can point at, and the `.tar.gz` archives the notebooks
download, with their SHA-256 hashes:

```bash
uv run python scripts/package_lab_assets.py \
  --run-dir outputs/ablations/blueprint_v2_final_d \
  --tokenizer /path/to/token_cache/tokenizer.json \
  --out dist/lab
export TFM_V2_CHECKPOINT=$PWD/dist/lab/blueprint_v2_lab_tuned
```

## On several GPUs

`config.scale.yaml` is the ~100M-parameter model: hidden 512, depths
3/16/12. Its `training.backend: ddp` gives every rank a full replica and
averages gradients. `training.batch_size` is per rank.

```bash
uv run torchrun --nproc-per-node N -m tfm.cli.main report --recipe blueprint_v2 \
  --config recipes/blueprint_v2/config.scale.yaml \
  --run-dir outputs/blueprint_v2/<run-id> --device cuda
```

## What a run writes

| File | Answers |
|---|---|
| `resolved_config.yaml` | exactly what ran |
| `split_manifest.json` | cutoffs and label windows |
| `probe_manifest.json` | users and positives per task, and skipped tasks with the reason |
| `metrics.md` | every arm, with under-supported rows flagged |
| `finetune_manifest.json` | adapted modules, trainable fraction, and the validation curve |
| `model_card.md` | composition, data, and limitations |

## Swap the dataset, or a slot

The model never sees pragmatiq's schema, only `TransactionEvent`s and
`EntityProfile`s. Another dataset needs an adapter that yields them, plus
label rows for its tasks ([data contract](../../docs/concepts/data-contract.md)).
The tokenizer is refitted on the new train side, and the checkpoint is
retrained. A checkpoint is only evaluated on the dataset it was pretrained on
([dataset strategy](../../docs/reference/datasets.md)).

Every slot is one config key, so a comparison changes one line. The
[building-block ablation](../../benchmarks/ablations/building_blocks)
does this on shared splits and writes one table.

## Where this lives in the kit

This notebook explains; the recipe runs. Every cell calls `src/tfm`, so the
same code runs in [`recipes/blueprint_v2`](../../recipes/blueprint_v2), and no
reusable logic lives here.